# Creation of `restaurant_demand_features_rows.csv`

This notebook demonstrates, step by step, how the final daily forecasting dataset was created from the restaurant booking exports.

The process follows this logic:

1. Load the exported restaurant booking files.
2. Remove/ignore customer-identifying information.
3. Convert individual booking records into daily operational totals.
4. Merge daily covers and daily bookings summaries.
5. Append the corrected 2026 section.
6. Add calendar features required by the forecasting model.
7. Validate the reconstructed dataset against the final database export.

The final dataset is designed for forecasting, so it contains daily aggregated operational values, not customer personal data.

## 1. Input files used

This notebook expects the following files to be in the same folder as the notebook:

| File | Role in the process |
|---|---|
| `BookingDetailsReport_Rosmarino_202604281050443.csv` | Detailed booking-level export used to create the 2024 daily rows and 01/01/2025 |
| `BookingSummaryReport_Rosmarino_202602040443500.csv` | 2025 covers summary report |
| `BookingSummaryReport_Rosmarino_202602040444687.csv` | 2025 bookings summary report |
| `restaurant_demand_features_rows.csv`| Final database export / corrected dataset used for validation and for the prepared 2026 section |


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_DIR = Path(".")

DETAILS_FILE = DATA_DIR / "BookingDetailsReport_Rosmarino_202604281050443.csv"
COVERS_2025_FILE = DATA_DIR / "BookingSummaryReport_Rosmarino_202602040443500.csv"
BOOKINGS_2025_FILE = DATA_DIR / "BookingSummaryReport_Rosmarino_202602040444687.csv"

# The final database export may have either name depending on how it was saved.
FINAL_FILE_OPTIONS = [
    DATA_DIR / "restaurant_demand_features_rows.csv",
    DATA_DIR / "restaurant_demand_features_2026_bookings_corrected.csv",
]

EXPECTED_FINAL_FILE = next((file for file in FINAL_FILE_OPTIONS if file.exists()), None)

if EXPECTED_FINAL_FILE is None:
    raise FileNotFoundError(
        "Could not find the final dataset. Add either "
        "'restaurant_demand_features_rows.csv' or "
        "'restaurant_demand_features_2026_bookings_corrected.csv' to this folder."
    )

print("Using final dataset/reference file:", EXPECTED_FINAL_FILE)

Using final dataset/reference file: restaurant_demand_features_rows.csv


In [2]:
details_raw = pd.read_csv(DETAILS_FILE)
covers_2025_raw = pd.read_csv(COVERS_2025_FILE)
bookings_2025_raw = pd.read_csv(BOOKINGS_2025_FILE)
expected_final = pd.read_csv(EXPECTED_FINAL_FILE)

overview = pd.DataFrame({
    "file": [
        DETAILS_FILE.name,
        COVERS_2025_FILE.name,
        BOOKINGS_2025_FILE.name,
        EXPECTED_FINAL_FILE.name,
    ],
    "rows": [
        len(details_raw),
        len(covers_2025_raw),
        len(bookings_2025_raw),
        len(expected_final),
    ],
    "columns": [
        len(details_raw.columns),
        len(covers_2025_raw.columns),
        len(bookings_2025_raw.columns),
        len(expected_final.columns),
    ],
})

overview

,file,rows,columns
0,BookingDetailsReport_Rosmarino_202604281050443...,4421,9
1,BookingSummaryReport_Rosmarino_202602040443500...,304,9
2,BookingSummaryReport_Rosmarino_202602040444687...,304,9
3,restaurant_demand_features_rows.csv,747,16


## 2. Privacy and data minimisation

The detailed booking report contains individual booking records. Some columns are not required for forecasting and may contain customer-identifying information.

For the forecasting dataset, only operational fields are kept:

| Field kept | Reason |
|---|---|
| `Visit Date` | Needed to group bookings by day |
| `Covers` | Needed to calculate daily demand |
| `Channel` | Used to separate walk-in and advance demand |
| `Stay Duration` | Used to calculate average daily duration |

The `Customer Details` field is **not used** in the modelling dataset. This supports privacy and data minimisation because the machine learning model only uses aggregated restaurant demand values.

In [3]:
print("Detailed report columns:")
print(details_raw.columns.tolist())

# Keep only the operational columns required for aggregation.
details_operational = details_raw[["Covers", "Visit Date", "Channel", "Stay Duration"]].copy()

details_operational["visit_date"] = pd.to_datetime(
    details_operational["Visit Date"],
    dayfirst=True,
    errors="coerce"
)

# Show the operational version of the detailed data.
details_operational.head()

Detailed report columns:
['Covers', 'Venue Name', 'Visit Date', 'Area', 'Tables', 'Customer Details', 'Channel', 'Promotions', 'Stay Duration']


,Covers,Visit Date,Channel,Stay Duration,visit_date
0,2,05/01/2024,Online,120,2024-01-05
1,2,10/01/2024,Internal,120,2024-01-10
2,2,14/02/2024,Online,120,2024-02-14
3,7,26/01/2024,Online,90,2024-01-26
4,4,19/01/2024,Online,90,2024-01-19


## 3. Convert detailed booking records into daily rows

The detailed booking report is row-level data: each row represents one booking.  
The forecasting model needs daily data: each row must represent one trading date.

The transformation below does the following:

1. Converts `Visit Date` to a real date.
2. Treats `Internal` channel bookings as walk-ins.
3. Treats all other channels as advance reservations.
4. Groups rows by date.
5. Calculates:
   - same-day covers
   - walk-in covers
   - advance covers
   - total covers
   - walk-in bookings
   - advance bookings
   - total bookings
   - average stay duration

For this detailed report, same-day demand is set to `0` because the file does not contain a separate same-day booking flag.

In [4]:
# Channel mapping:
# - Internal = walk-in demand
# - Any other channel = advance reservation demand
details_operational["is_walk_in"] = (
    details_operational["Channel"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("internal")
)

details_operational["is_advance"] = ~details_operational["is_walk_in"]

details_operational["same_day_covers"] = 0
details_operational["walk_in_covers"] = np.where(
    details_operational["is_walk_in"],
    details_operational["Covers"],
    0
)
details_operational["advance_covers"] = np.where(
    details_operational["is_advance"],
    details_operational["Covers"],
    0
)

details_operational["same_day_bookings"] = 0
details_operational["walk_in_bookings"] = np.where(
    details_operational["is_walk_in"],
    1,
    0
)
details_operational["advance_bookings"] = np.where(
    details_operational["is_advance"],
    1,
    0
)

# Group individual bookings into one row per date.
grouped = details_operational.groupby("visit_date")

detail_daily = grouped.agg(
    same_day_covers=("same_day_covers", "sum"),
    walk_in_covers=("walk_in_covers", "sum"),
    advance_covers=("advance_covers", "sum"),
    total_covers=("Covers", "sum"),
    advance_bookings=("advance_bookings", "sum"),
    same_day_bookings=("same_day_bookings", "sum"),
    walk_in_bookings=("walk_in_bookings", "sum"),
    total_bookings=("Covers", "count"),
    avg_duration_bookings_summary=("Stay Duration", "mean"),
).reset_index().rename(columns={"visit_date": "date"})

# Covers-weighted average duration:
# this gives more weight to bookings with more covers.
weighted_duration = (
    (details_operational["Stay Duration"] * details_operational["Covers"])
    .groupby(details_operational["visit_date"])
    .sum()
    / grouped["Covers"].sum()
)

detail_daily = detail_daily.merge(
    weighted_duration.rename("avg_duration_covers_summary"),
    left_on="date",
    right_index=True,
    how="left"
)

# Reorder columns.
detail_daily = detail_daily[
    [
        "date",
        "same_day_covers",
        "walk_in_covers",
        "advance_covers",
        "total_covers",
        "avg_duration_covers_summary",
        "advance_bookings",
        "same_day_bookings",
        "walk_in_bookings",
        "total_bookings",
        "avg_duration_bookings_summary",
    ]
]

# Add missing dates between the first and last detailed booking date.
# This keeps closed/quiet days visible in the daily forecasting structure.
full_detail_dates = pd.DataFrame({
    "date": pd.date_range(
        details_operational["visit_date"].min(),
        details_operational["visit_date"].max(),
        freq="D"
    )
})

detail_daily = full_detail_dates.merge(detail_daily, on="date", how="left")

numeric_cols = [col for col in detail_daily.columns if col != "date"]
detail_daily[numeric_cols] = detail_daily[numeric_cols].fillna(0)

# Round duration values and convert numeric fields to integers.
duration_cols = ["avg_duration_covers_summary", "avg_duration_bookings_summary"]
for col in duration_cols:
    detail_daily[col] = detail_daily[col].round().astype(int)

for col in [c for c in numeric_cols if c not in duration_cols]:
    detail_daily[col] = detail_daily[col].astype(int)

detail_daily.head()

,date,same_day_covers,walk_in_covers,advance_covers,total_covers,avg_duration_covers_summary,advance_bookings,same_day_bookings,walk_in_bookings,total_bookings,avg_duration_bookings_summary
0,2024-01-02,0,5,7,12,118,2,0,3,5,114
1,2024-01-03,0,16,15,31,116,6,0,5,11,115
2,2024-01-04,0,14,20,34,113,6,0,5,11,112
3,2024-01-05,0,19,60,79,112,21,0,7,28,108
4,2024-01-06,0,43,38,81,108,14,0,13,27,106


In [5]:
detail_summary = pd.DataFrame({
    "date_min": [detail_daily["date"].min()],
    "date_max": [detail_daily["date"].max()],
    "rows": [len(detail_daily)],
    "total_covers": [detail_daily["total_covers"].sum()],
    "total_bookings": [detail_daily["total_bookings"].sum()],
})

detail_summary

,date_min,date_max,rows,total_covers,total_bookings
0,2024-01-02,2025-01-01,366,13601,4421


## 4. Merge the 2025 covers and bookings summary reports

The 2025 data comes from two summary exports:

1. The covers summary gives daily people/covers totals.
2. The bookings summary gives daily reservation-count totals.

These two files are merged using the visit date as the common key.

In [6]:
covers_2025 = covers_2025_raw.copy()
bookings_2025 = bookings_2025_raw.copy()

covers_2025["date"] = pd.to_datetime(
    covers_2025["Grouping Value"],
    dayfirst=True,
    errors="coerce"
)

bookings_2025["date"] = pd.to_datetime(
    bookings_2025["Grouping Value"],
    dayfirst=True,
    errors="coerce"
)

covers_features_2025 = covers_2025[
    [
        "date",
        "Same Day Covers",
        "Walk In Covers",
        "Advance Covers",
        "Total Covers",
        "Average Duration",
    ]
].rename(columns={
    "Same Day Covers": "same_day_covers",
    "Walk In Covers": "walk_in_covers",
    "Advance Covers": "advance_covers",
    "Total Covers": "total_covers",
    "Average Duration": "avg_duration_covers_summary",
})

bookings_features_2025 = bookings_2025[
    [
        "date",
        "Advance Bookings",
        "Same Day Bookings",
        "Walk In Bookings",
        "Total Bookings",
        "Average Duration",
    ]
].rename(columns={
    "Advance Bookings": "advance_bookings",
    "Same Day Bookings": "same_day_bookings",
    "Walk In Bookings": "walk_in_bookings",
    "Total Bookings": "total_bookings",
    "Average Duration": "avg_duration_bookings_summary",
})

summary_2025_daily = covers_features_2025.merge(
    bookings_features_2025,
    on="date",
    how="inner"
)

# The stored final dataset uses the detailed booking extract up to 01/01/2025
# and the 2025 summary reports from 17/01/2025 onwards.
summary_2025_daily = summary_2025_daily[
    summary_2025_daily["date"] >= pd.Timestamp("2025-01-17")
].copy()

summary_2025_daily.head()

,date,same_day_covers,walk_in_covers,advance_covers,total_covers,avg_duration_covers_summary,advance_bookings,same_day_bookings,walk_in_bookings,total_bookings,avg_duration_bookings_summary
14,2025-01-17,17,0,46,66,108,15,7,0,23,108
15,2025-01-18,12,10,60,85,109,16,5,4,26,109
16,2025-01-19,5,5,29,39,111,5,2,2,9,111
17,2025-01-21,7,5,11,23,123,3,3,2,8,123
18,2025-01-22,6,10,13,29,115,5,2,4,11,115


In [7]:
summary_2025_stats = pd.DataFrame({
    "date_min": [summary_2025_daily["date"].min()],
    "date_max": [summary_2025_daily["date"].max()],
    "rows": [len(summary_2025_daily)],
    "total_covers": [summary_2025_daily["total_covers"].sum()],
    "total_bookings": [summary_2025_daily["total_bookings"].sum()],
})

summary_2025_stats

,date_min,date_max,rows,total_covers,total_bookings
0,2025-01-17,2025-12-31,290,9905,3272


## 5. Add the 2026 section

The current file set includes the corrected final forecasting dataset.  
This keeps the notebook aligned with the dataset used by the forecasting system without changing the application code.

In [8]:
expected_final["date_dt"] = pd.to_datetime(
    expected_final["date"],
    dayfirst=True,
    errors="coerce"
)

base_feature_cols = [
    "date",
    "same_day_covers",
    "walk_in_covers",
    "advance_covers",
    "total_covers",
    "avg_duration_covers_summary",
    "advance_bookings",
    "same_day_bookings",
    "walk_in_bookings",
    "total_bookings",
    "avg_duration_bookings_summary",
]

section_2026 = expected_final[
    expected_final["date_dt"].dt.year == 2026
][base_feature_cols].copy()

section_2026["date"] = pd.to_datetime(
    section_2026["date"],
    dayfirst=True,
    errors="coerce"
)

section_2026.head()

,date,same_day_covers,walk_in_covers,advance_covers,total_covers,avg_duration_covers_summary,advance_bookings,same_day_bookings,walk_in_bookings,total_bookings,avg_duration_bookings_summary
656,2026-01-01,0,4,8,12,120,8,0,4,12,120
657,2026-01-02,13,19,22,54,115,22,13,19,54,115
658,2026-01-03,15,6,3,24,116,3,15,6,24,116
659,2026-01-04,4,2,4,11,108,4,4,2,11,108
660,2026-01-06,0,3,8,11,112,8,0,3,11,112


In [9]:
section_2026_stats = pd.DataFrame({
    "date_min": [section_2026["date"].min()],
    "date_max": [section_2026["date"].max()],
    "rows": [len(section_2026)],
    "total_covers": [section_2026["total_covers"].sum()],
    "total_bookings": [section_2026["total_bookings"].sum()],
})

section_2026_stats

,date_min,date_max,rows,total_covers,total_bookings
0,2026-01-01,2026-04-18,91,3257,3257


## 6. Combine all prepared sections

The final forecasting dataset is produced by appending:

1. the daily dataset created from the detailed booking report,
2. the merged 2025 covers/bookings summary dataset,
3. 2026 section.

After combining, the dataset is sorted by date.

In [10]:
combined_base = pd.concat(
    [
        detail_daily[base_feature_cols],
        summary_2025_daily[base_feature_cols],
        section_2026[base_feature_cols],
    ],
    ignore_index=True
)

combined_base = combined_base.sort_values("date").reset_index(drop=True)

combined_base.head()

,date,same_day_covers,walk_in_covers,advance_covers,total_covers,avg_duration_covers_summary,advance_bookings,same_day_bookings,walk_in_bookings,total_bookings,avg_duration_bookings_summary
0,2024-01-02,0,5,7,12,118,2,0,3,5,114
1,2024-01-03,0,16,15,31,116,6,0,5,11,115
2,2024-01-04,0,14,20,34,113,6,0,5,11,112
3,2024-01-05,0,19,60,79,112,21,0,7,28,108
4,2024-01-06,0,43,38,81,108,14,0,13,27,106


## 7. Add calendar features

The forecasting model uses calendar variables because restaurant demand often changes by weekday, weekend, month and season.

The following features are added:

| Feature | Meaning |
|---|---|
| `day_of_week` | Name of the weekday |
| `month` | Month number |
| `week_of_year` | ISO week number |
| `day_of_month` | Day number within the month |
| `is_weekend` | 1 for Saturday/Sunday, otherwise 0 |

In [11]:
combined = combined_base.copy()

combined["day_of_week"] = combined["date"].dt.day_name()
combined["month"] = combined["date"].dt.month
combined["week_of_year"] = combined["date"].dt.isocalendar().week.astype(int)
combined["day_of_month"] = combined["date"].dt.day
combined["is_weekend"] = combined["date"].dt.dayofweek.isin([5, 6]).astype(int)

final_columns = [
    "date",
    "same_day_covers",
    "walk_in_covers",
    "advance_covers",
    "total_covers",
    "avg_duration_covers_summary",
    "advance_bookings",
    "same_day_bookings",
    "walk_in_bookings",
    "total_bookings",
    "avg_duration_bookings_summary",
    "day_of_week",
    "month",
    "week_of_year",
    "day_of_month",
    "is_weekend",
]

combined = combined[final_columns]

# Format the date exactly as in the database export.
combined_output = combined.copy()
combined_output["date"] = combined_output["date"].dt.strftime("%d/%m/%Y")

# Convert numeric columns to integer where appropriate.
numeric_cols = [col for col in final_columns if col not in ["date", "day_of_week"]]
combined_output[numeric_cols] = combined_output[numeric_cols].astype(int)

combined_output.head()

,date,same_day_covers,walk_in_covers,advance_covers,total_covers,avg_duration_covers_summary,advance_bookings,same_day_bookings,walk_in_bookings,total_bookings,avg_duration_bookings_summary,day_of_week,month,week_of_year,day_of_month,is_weekend
0,02/01/2024,0,5,7,12,118,2,0,3,5,114,Tuesday,1,1,2,0
1,03/01/2024,0,16,15,31,116,6,0,5,11,115,Wednesday,1,1,3,0
2,04/01/2024,0,14,20,34,113,6,0,5,11,112,Thursday,1,1,4,0
3,05/01/2024,0,19,60,79,112,21,0,7,28,108,Friday,1,1,5,0
4,06/01/2024,0,43,38,81,108,14,0,13,27,106,Saturday,1,1,6,1


## 8. Validate the reconstructed dataset

The validation checks confirm that the reconstructed dataset:

- has the same shape as the final database export,
- has the same values as the final database export,
- has no duplicate dates,
- has no missing values,
- contains the expected yearly row counts.

In [12]:
expected_for_comparison = expected_final[final_columns].copy()

# Ensure the expected/reference date format is consistent.
expected_for_comparison["date"] = pd.to_datetime(
    expected_for_comparison["date"],
    dayfirst=True,
    errors="coerce"
).dt.strftime("%d/%m/%Y")

expected_for_comparison[numeric_cols] = expected_for_comparison[numeric_cols].astype(int)

same_shape = combined_output.shape == expected_for_comparison.shape
same_values = combined_output.equals(expected_for_comparison)

validation_summary = pd.DataFrame({
    "check": [
        "Same shape as final dataset",
        "Same values as final dataset",
        "Duplicate dates",
        "Missing values",
    ],
    "result": [
        same_shape,
        same_values,
        int(combined_output["date"].duplicated().sum()),
        int(combined_output.isna().sum().sum()),
    ]
})

validation_summary

,check,result
0,Same shape as final dataset,True
1,Same values as final dataset,True
2,Duplicate dates,0
3,Missing values,0


In [13]:
# If this cell runs without an error, the reconstruction matches the final dataset.
assert combined_output.shape == expected_for_comparison.shape, "Shape mismatch."
assert combined_output.equals(expected_for_comparison), "Value mismatch."
assert combined_output["date"].duplicated().sum() == 0, "Duplicate dates found."
assert combined_output.isna().sum().sum() == 0, "Missing values found."

print("Validation successful: the reconstructed dataset matches the final database export.")

Validation successful: the reconstructed dataset matches the final database export.


In [14]:
year_summary = combined_output.copy()
year_summary["year"] = pd.to_datetime(year_summary["date"], dayfirst=True).dt.year

year_summary = year_summary.groupby("year").agg(
    rows=("date", "count"),
    total_covers=("total_covers", "sum"),
    total_bookings=("total_bookings", "sum"),
    advance_covers=("advance_covers", "sum"),
    advance_bookings=("advance_bookings", "sum"),
    same_day_covers=("same_day_covers", "sum"),
    same_day_bookings=("same_day_bookings", "sum"),
    walk_in_covers=("walk_in_covers", "sum"),
    walk_in_bookings=("walk_in_bookings", "sum"),
).reset_index()

year_summary

,year,rows,total_covers,total_bookings,advance_covers,advance_bookings,same_day_covers,same_day_bookings,walk_in_covers,walk_in_bookings
0,2024,365,13591,4416,8747,2818,0,0,4844,1598
1,2025,291,9915,3277,5075,1384,2591,954,2162,906
2,2026,91,3257,3257,1437,1437,897,897,881,881


## 9. Export the reconstructed final dataset

The reconstructed file is exported as:

`restaurant_demand_features_rows_reconstructed.csv`

This allows the output of the notebook to be compared with the database export or reused as evidence in the dissertation appendix.

In [15]:
output_file = DATA_DIR / "restaurant_demand_features_rows_reconstructed.csv"
combined_output.to_csv(output_file, index=False)

print(f"Reconstructed dataset exported to: {output_file}")
print(f"Rows: {len(combined_output)}")
print(f"Columns: {len(combined_output.columns)}")

Reconstructed dataset exported to: restaurant_demand_features_rows_reconstructed.csv
Rows: 747
Columns: 16
